# Validation tuning — seed 0

GPU0: GL lambda0.1,1,10. GPU1: MambaCL+GL lambda0.1,1,10. Each candidate uses the first three tasks, 10 epochs (LR milestones5/8), stratified10% training holdout. Max validation AIA, ties use smaller lambda. Repeat with previous Output until COMPLETE.

Same setup flow as H4: Internet ON, Kaggle Secret `github_token`, T4 x2. Source is cloned from GitHub and checked out at a fixed commit. Attach official pretrained weights. Original ImageNet-R images or imagenet-r.tar are optional Inputs: all Input files are searched automatically. If filenames have been changed (as in my1nonly/imagenet-r), the notebook automatically downloads the author archive (about2.19GB) with Internet ON. It preserves the exact Mamba-CL24000/6000 split and200 classes; it never guesses a mapping from renamed files. Previous Output is for resume and for carrying01 selection into02. Checkpoints are compact; large immutable contexts/curvature can remain on earlier Output Inputs. Keep the shards listed in resume_dependencies.json attached. Archives are optional; the default saves the Output tree directly. No source ZIP is required.

Every5 optimizer steps, live output shows throughput, peak VRAM, loss, GPU utilization and exposed dataloader wait. progress.json updates each step; resume still uses the last committed epoch. Heartbeats show /kaggle/working usage and request PAUSED below2GiB free. Before01 launches either lane, an isolated20-step MambaCL probe on GPU0 must reach10 images/s; slower results stop for review. CPU checks pass; actual GPU throughput remains to be measured on Kaggle.

In [ ]:
BRANCH = 'GL'
REPO_COMMIT = 'caf321bca9bd8296a6f8a1c61af4b78619723b6e'
RUN_TAG = 'glcl_imagenetr_s0_caf321bca9bd'
STAGE = 'tune'
DATA_ROOT = None  # optional raw OR official train/val root; auto-discovered under /kaggle/input
PRETRAINED = None  # optional explicit path to defocus_mamba_large_cls_21k.pth
JOBS = [[{'name': 'GL_lambda0.1', 'arm': 'GL', 'strength': 0.1, 'stage': 'tune'}, {'name': 'GL_lambda1', 'arm': 'GL', 'strength': 1.0, 'stage': 'tune'}, {'name': 'GL_lambda10', 'arm': 'GL', 'strength': 10.0, 'stage': 'tune'}], [{'name': 'MambaCL+GL_lambda0.1', 'arm': 'MambaCL+GL', 'strength': 0.1, 'stage': 'tune'}, {'name': 'MambaCL+GL_lambda1', 'arm': 'MambaCL+GL', 'strength': 1.0, 'stage': 'tune'}, {'name': 'MambaCL+GL_lambda10', 'arm': 'MambaCL+GL', 'strength': 10.0, 'stage': 'tune'}]]
import time
SESSION_START = time.time()

In [ ]:
STORAGE_PREFLIGHT = '"""Recover generated scratch space without deleting valid experiment checkpoints."""\nimport argparse\nimport json\nimport os\nimport pickletools\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\n\ndef checkpoint_position(path):\n    # Bootstrap is deliberately stdlib-only, before the pinned repo/venv exists.\n    with zipfile.ZipFile(path) as archive:\n        name = next(n for n in archive.namelist() if n.endswith(\'/data.pkl\'))\n        with archive.open(name) as stream:\n            prefix = stream.read(256*1024)\n    fields, pending = {}, None\n    for op, arg, _ in pickletools.genops(prefix):\n        if op.name in (\'BINUNICODE\', \'SHORT_BINUNICODE\', \'UNICODE\') and arg in (\'task\', \'epoch\', \'task_complete\'):\n            pending = arg\n        elif pending and op.name in (\'BININT\', \'BININT1\', \'BININT2\', \'INT\', \'NEWTRUE\', \'NEWFALSE\'):\n            fields[pending] = True if op.name == \'NEWTRUE\' else False if op.name == \'NEWFALSE\' else arg\n            pending = None\n            if len(fields) == 3:\n                return (int(fields[\'task\']), int(fields[\'epoch\']), int(fields[\'task_complete\']))\n    raise ValueError(\'Missing checkpoint position\')\n\n\ndef active_gl_processes():\n    found = []\n    proc = Path(\'/proc\')\n    if proc.is_dir():\n        for path in proc.iterdir():\n            if not path.name.isdigit():\n                continue\n            try:\n                words = (path / \'cmdline\').read_bytes().split(b\'\\0\')\n            except OSError:\n                continue\n            if any(words[i:i+2] in ([b\'-m\', b\'glcl.runner\'], [b\'-m\', b\'glcl.kaggle\']) for i in range(len(words)-1)):\n                found.append(int(path.name))\n    return found\n\n\ndef migrate_directory(source, destination, events):\n    if not source.is_dir() or source.resolve() == destination.resolve():\n        return\n    if destination.exists():\n        # Never overwrite another environment/cache. Retain the old one under\n        # ephemeral storage so the new canonical cache can be validated first.\n        alternate = destination.with_name(destination.name + \'-previous-working\')\n        if alternate.exists():\n            raise RuntimeError(\'Both migration targets exist; inspect storage manually: \' + str(alternate))\n        destination = alternate\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    shutil.move(str(source), str(destination))\n    events.append(dict(moved_from=str(source), moved_to=str(destination)))\n    # Pip is invoked as python -m pip. Repair console-script shebangs as well.\n    if (destination / \'pyvenv.cfg\').is_file():\n        for file in (destination / \'bin\').iterdir():\n            if file.is_file() and not file.is_symlink():\n                data = file.read_bytes()\n                if data.startswith((\'#!\' + str(source)).encode()):\n                    file.write_bytes(data.replace(str(source).encode(), str(destination).encode(), 1))\n\n\ndef prepare(working, ephemeral, repo=None):\n    working, ephemeral = Path(working).absolute(), Path(ephemeral).absolute()\n    running = active_gl_processes()\n    if running:\n        raise RuntimeError(\'A previous GL launcher/worker is still active (PIDs \' + str(running)\n                           + \'). Stop the failed run before repairing its scratch files.\')\n    output = working / \'glcl_outputs\'\n    output.mkdir(parents=True, exist_ok=True)\n    ephemeral.mkdir(parents=True, exist_ok=True)\n    before = shutil.disk_usage(working).free\n    removed, events, recovered = [], [], []\n    paths = [p for p in output.rglob(\'*.tmp\')\n             if p.name.endswith((\'.pt.tmp\', \'.json.tmp\'))]\n    paths += list(output.glob(\'*/*/contexts/context-pending.pt\'))\n    paths.append(working / \'glcl_outputs.tar.tmp\')\n    for path in paths:\n        if path.is_file() or path.is_symlink():\n            if path.name == \'latest.pt.tmp\' and zipfile.is_zipfile(path):\n                try:\n                    rank = checkpoint_position(path)\n                    with zipfile.ZipFile(path) as archive:\n                        if archive.testzip() is not None:\n                            raise ValueError(\'Temporary checkpoint CRC failed\')\n                    current = path.with_suffix(\'\')\n                    try:\n                        old_rank = checkpoint_position(current)\n                    except (OSError, ValueError, zipfile.BadZipFile, StopIteration):\n                        old_rank = (-1, -1, -1)\n                    if rank > old_rank:\n                        os.replace(path, current)\n                        recovered.append(dict(path=str(current), task=rank[0]+1, epoch=rank[1]))\n                        continue\n                except (OSError, ValueError, zipfile.BadZipFile, StopIteration):\n                    pass\n            size = path.stat().st_size if path.exists() else 0\n            path.unlink()\n            removed.append(dict(path=str(path), bytes=size))\n    # The downloaded tar is a reproducible cache, not a checkpoint. Remove it\n    # only if every official image has already been extracted and is present.\n    tools = Path(repo)/\'upstream/mamba-cl/tools\' if repo else None\n    if tools and all((tools/f\'imagenet_r_{mode}.txt\').is_file() for mode in (\'train\', \'val\')):\n        names = [line for mode in (\'train\', \'val\') for line in (tools/f\'imagenet_r_{mode}.txt\').read_text().splitlines()]\n        for base in (working/\'input_data/original_imagenet_r\', ephemeral/\'input_data/original_imagenet_r\'):\n            archive = base/\'imagenet-r.tar\'\n            if archive.is_file() and len(names) == 30000 and all((base/\'images\'/name).is_file() for name in names):\n                size = archive.stat().st_size\n                archive.unlink()\n                removed.append(dict(path=str(archive), bytes=size, reason=\'Verified extracted original images; redundant download cache\'))\n    legacy_ephemeral = working.parent.parent / \'temp/glcl\'\n    if legacy_ephemeral.resolve() != ephemeral.resolve():\n        for name in (\'env\', \'input_data\', \'cache\', \'python\', \'legacy_resume\'):\n            migrate_directory(legacy_ephemeral/name, ephemeral/name, events)\n    migrate_directory(working.parent / \'glcl_env\', ephemeral / \'env\', events)\n    migrate_directory(working / \'input_data\', ephemeral / \'input_data\', events)\n    result = dict(removed_failed_temporary_files=removed, recovered_complete_temporary_checkpoints=recovered, migrations=events,\n                  preserved=\'All latest.pt files, immutable contexts/curvature and valid archives\',\n                  working_free_bytes_before=before, working_free_bytes_after=shutil.disk_usage(working).free,\n                  ephemeral_free_bytes=shutil.disk_usage(ephemeral).free)\n    target = output / \'storage_preflight.json\'\n    target.write_text(json.dumps(result, indent=2) + \'\\n\')\n    print(json.dumps(result, indent=2))\n    return result\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--working\', required=True)\n    p.add_argument(\'--ephemeral\', required=True)\n    p.add_argument(\'--repo\')\n    a = p.parse_args()\n    prepare(a.working, a.ephemeral, a.repo)\n\n\nif __name__ == \'__main__\':\n    main()\n'

import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

os.environ.update(TQDM_DISABLE='1', PYTHONUNBUFFERED='1',
                  PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION='python')
MAIN_URL = 'https://github.com/duyh80456-code/Gramian-Laplace-GL-cho-continual-learning-tr-n-Mamba.git'
WORKING = Path('/kaggle/working')
# A previous execution leaves the kernel inside REPO. Always leave that tree
# before removing it, and also recover if a prior run already deleted the cwd.
WORKING.mkdir(parents=True, exist_ok=True)
os.chdir(WORKING)
REPO = Path('/kaggle/working/gramian-laplace')
RUN_ROOT = Path('/kaggle/working/glcl')
EPHEMERAL = Path('/tmp/glcl')
os.environ.update(GL_EPHEMERAL_ROOT=str(EPHEMERAL), UV_CACHE_DIR=str(EPHEMERAL/'cache/uv'),
                  PIP_CACHE_DIR=str(EPHEMERAL/'cache/pip'), TMPDIR=str(EPHEMERAL/'tmp'),
                  UV_PYTHON_INSTALL_DIR=str(EPHEMERAL/'python'), GL_MIN_FREE_BYTES=str(2*1024**3))
(EPHEMERAL/'tmp').mkdir(parents=True, exist_ok=True)
OUTPUT = RUN_ROOT / 'glcl_outputs'
ROOT_OUT = OUTPUT
subprocess.run([sys.executable, '-c', STORAGE_PREFLIGHT, '--working', str(RUN_ROOT),
                '--ephemeral', str(EPHEMERAL), '--repo', str(REPO)], cwd=WORKING, check=True)
# This scientific protocol must not overwrite or resume old fp32 experiments.
if (OUTPUT/'tune').exists() or (OUTPUT/'main').exists():
    identities=list(OUTPUT.glob('*/*/identity.json'))
    if any(json.loads(p.read_text()).get('config',{}).get('protocol') != 'amp_fp16_resize256_tune10_v2' for p in identities):
        previous=RUN_ROOT/('glcl_outputs_before_amp_'+str(int(time.time())))
        OUTPUT.rename(previous)
        print('Preserved old protocol Output at:',previous)
OUTPUT.mkdir(parents=True,exist_ok=True)

def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    temporary.replace(path)

def private_clone(url, destination, commit):
    token = UserSecretsClient().get_secret('github_token').strip()
    if not token:
        raise RuntimeError('Enable Kaggle Secret github_token for this notebook')
    askpass = Path('/kaggle/working/.glcl_git_askpass.py')
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = dict(os.environ, GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0')
    try:
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', url, str(destination)], cwd=destination.parent, env=env, check=True)
        subprocess.run(['git', '-C', str(destination), 'fetch', 'origin', commit], cwd=destination.parent, env=env, check=True)
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], cwd=destination.parent, env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

# Only refresh the source checkout. Outputs and environment live elsewhere.
if REPO.exists():
    shutil.rmtree(REPO)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], cwd=WORKING, text=True).strip()
if CODE_COMMIT != REPO_COMMIT:
    raise RuntimeError('Pinned repository revision mismatch')
os.chdir(REPO)
print('Resolved code commit:', CODE_COMMIT, 'pin:', REPO_COMMIT)
print('Official MambaCL:', json.loads((REPO/'upstream/LOCK.json').read_text()))

In [ ]:
# Isolated Python 3.11 + official torch 2.2.2; no Kaggle host torch replacement.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.5.31'], check=True)
VENV = EPHEMERAL/'env'
if not (VENV/'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
                '-r', str(REPO/'requirements-kaggle.txt'), 'pip==24.2'], check=True)

ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', PYTHONUNBUFFERED='1', TQDM_DISABLE='1')

In [ ]:
# Recompute the supervisor-corrected section-six checks, same fixture/noises:
# C .852433 > proxy .725509 and >=.8; Delta .968065 > proxy .950543 and >=.85.
# Test5 remains DEFERRED until phase two.
# A failure intentionally stops before model initialization/training.
subprocess.run([PYTHON, '-m', 'tests.section6'], env=ENV, check=True)
print((REPO/'reports/section6.json').read_text())

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
subprocess.run([PYTHON, '-c', "from glcl.kaggle import restore; restore('/kaggle/input', '/kaggle/working/glcl/glcl_outputs')"], env=ENV, check=True)
commit_path = OUTPUT/'code_commit.json'
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append(dict(timestamp=datetime.now(timezone.utc).isoformat(), pin=REPO_COMMIT, commit=CODE_COMMIT, stage=STAGE))
atomic_json(sessions, commit_path)
DATA_REPORT = OUTPUT/'data_preparation.json'
prepare = [PYTHON, 'scripts/prepare_imagenet_r.py', '--input', str(DATA_ROOT or '/kaggle/input'),
           '--stage', str(EPHEMERAL/'input_data'/'imagenet_r'), '--report', str(DATA_REPORT), '--download-official']
subprocess.run(prepare, env=ENV, check=True)
prepared = json.loads(DATA_REPORT.read_text())
if prepared['status'] != 'PASSED':
    raise RuntimeError('ImageNet-R preparation failed; training blocked')
RAW_DATA_ROOT = prepared['data_dir']
DATA_ROOT = str(EPHEMERAL/'input_data'/'imagenet_r_resized256')
subprocess.run([PYTHON,'scripts/resize_imagenet_r.py','--source',RAW_DATA_ROOT,
                '--destination',DATA_ROOT],env=ENV,check=True)
prepared.update(training_data_dir=DATA_ROOT,resize=json.loads((Path(DATA_ROOT)/'resize_manifest.json').read_text()))
atomic_json(prepared,DATA_REPORT)
if PRETRAINED is None:
    PRETRAINED = subprocess.check_output(
        [PYTHON, 'scripts/prepare_imagenet_r.py', '--find-checkpoint', '/kaggle/input'],
        env=ENV, text=True).strip()
print('Data:', DATA_ROOT, 'Weights:', PRETRAINED)

In [ ]:
# Install official CUDA wheels for the isolated Torch/Python/ABI combination.
# No source compilation and no dependency resolution that could upgrade Torch.
subprocess.run([PYTHON, 'scripts/install_cuda_extensions.py',
                '--log', str(OUTPUT/'environment_install.log'),
                '--report', str(OUTPUT/'environment_install.json')], env=ENV, check=True)
ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1',
           PYTHONUNBUFFERED='1', TQDM_DISABLE='1')
subprocess.run([PYTHON, '-c', "import torch; print(torch.__version__); print([(i,torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]); assert torch.cuda.device_count()==2"], env=ENV, check=True)
# Syntax/import only; no image forward or end-to-end smoke test.
subprocess.run([PYTHON, '-c', "import sys; sys.path.insert(0,'upstream/mamba-cl'); import train_eval; from utils.mamba_block import is_fast_path_available; assert is_fast_path_available"], env=ENV, check=True)

In [ ]:
# Required throughput gate: fresh MambaCL,20 optimizer steps on GPU0 only.
BENCHMARK_OUT = OUTPUT/'throughput_probe'
probe = [PYTHON, 'scripts/benchmark_throughput.py', '--data', DATA_ROOT,
         '--pretrained', PRETRAINED, '--out', str(BENCHMARK_OUT)]
probe_result = subprocess.run(probe, env=ENV)
if probe_result.returncode != 0:
    raise RuntimeError('01 is PAUSED before tuning: inspect throughput_probe/benchmark.json and benchmark output. Below10 images/s requires review.')

In [ ]:
PERFORMANCE_OBSERVER = '"""Read-only metric follower for existing Mamba-CL logs; no model/data imports.\n\nThe notebook embeds this observer so its pinned training source and checkpoint\nidentity can remain unchanged. Validation is labeled only for the tuning stage.\n"""\nimport argparse\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport threading\nimport time\n\nNUMBER = r\'([+-]?(?:\\d+(?:\\.\\d*)?|\\.\\d+)(?:[eE][+-]?\\d+)?|[+-]?(?:nan|inf))\'\n\n\ndef value(line, key, percent=False):\n    match = re.search(r\'\\b\' + re.escape(key) + r\'\\s*=\\s*\' + NUMBER + (r\'\\s*%\' if percent else \'\'), line, re.I)\n    if not match:\n        return None\n    number = float(match.group(1))\n    return number if math.isfinite(number) else match.group(1)\n\n\ndef metric_event(line, stage):\n    line = line.strip()\n    if line.startswith(\'STEP_METRICS \'):\n        fields = json.loads(line[len(\'STEP_METRICS \'):])\n        return \'THROUGHPUT\', fields, line\n    score = \'val_acc\' if stage == \'tune\' else \'test_acc\'\n    task = re.search(r\'Task \\[\\s*(\\d+)\\s*/\\s*(\\d+)\\s*\\]\', line)\n    epoch = re.search(r\'Epoch \\[\\s*(\\d+)\\s*/\\s*(\\d+)\\s*\\]\', line)\n    fields = dict(task=int(task[1]), total_tasks=int(task[2])) if task else {}\n    if epoch and \'loss=\' in line:\n        fields.update(epoch=int(epoch[1]), total_epochs=int(epoch[2]),\n                      loss=value(line, \'loss\'), train_acc=value(line, \'acc_top1\', True))\n        return \'TRAIN\', fields, line.replace(\'acc_top1=\', \'train_acc=\')\n    if \'Results of task [\' in line:\n        fields.update(task=int(re.search(r\'Results of task \\[(\\d+)\\]\', line)[1]))\n        fields[score] = value(line, \'class_inc_last_acc\', True)\n        fields[\'forgetting\'] = value(line, \'class_inc_last_forg\', True)\n        return \'VALIDATION\' if stage == \'tune\' else \'TEST\', fields, line.replace(\'class_inc_last_acc=\', score + \'=\').replace(\'class_inc_last_forg=\', \'forgetting=\')\n    if \':: Eval [\' in line:\n        fields[score + \'_task\'] = value(line, \'acc_class_inc\', True)\n        return \'EVAL_TASK\', fields, line.replace(\'acc_class_inc=\', score + \'_task=\')\n    if \'Training time =\' in line or \'Time so far:\' in line:\n        return \'TIME\', fields, line\n    if \'GL statistics \' in line:\n        match = re.search(r\'GL statistics (\\d+)/(\\d+)\', line)\n        if match and (int(match[1]) % 100 == 0 or match[1] == match[2]):\n            return \'STATISTICS\', {}, line\n    if \'Diagnostics \' in line:\n        match = re.search(r\'Diagnostics (\\d+)/(\\d+)\', line)\n        if match and (int(match[1]) % 8 == 0 or match[1] == match[2]):\n            return \'DIAGNOSTICS\', {}, line\n    if line.startswith(\'Traceback\') or re.match(r\'^[\\w.]*(?:Error|Exception):\', line):\n        return \'ERROR\', {}, line\n    return None\n\n\nclass MetricFollower:\n    def __init__(self, root, queues):\n        self.root = Path(root)\n        self.started = time.monotonic()\n        self.states = {}\n        for gpu, queue in enumerate(queues):\n            for job in queue:\n                key = job[\'stage\'] + \'/\' + job[\'name\']\n                self.states[key] = dict(gpu=gpu, job=job[\'name\'], stage=job[\'stage\'],\n                                        offset=0, pending=b\'\', inode=None, initialized=False,\n                                        latest={}, summary_seen=False,\n                                        has_history=(self.root / key / \'console.log\').is_file(),\n                                        summary_history=(self.root / key / \'summary.json\').is_file())\n\n    def emit(self, state, event, historical=False):\n        kind, fields, text = event\n        elapsed = int(time.monotonic() - self.started)\n        stamp = f\'{elapsed//3600:02d}:{elapsed//60%60:02d}:{elapsed%60:02d}\'\n        origin = \'previous log\' if historical else \'live\'\n        print(f"[{stamp} GPU {state[\'gpu\']} | {state[\'job\']} | {kind} | {origin}] {text}", flush=True)\n        state[\'latest\'][kind] = dict(fields, log_line=text, from_previous_log=historical)\n\n    def poll(self):\n        changed = False\n        for key, state in self.states.items():\n            path = self.root / key / \'console.log\'\n            if path.is_file():\n                stat = path.stat()\n                inode = (stat.st_dev, stat.st_ino)\n                first = not state[\'initialized\']\n                if state[\'inode\'] != inode or stat.st_size < state[\'offset\']:\n                    state.update(offset=0, pending=b\'\', inode=inode)\n                with path.open(\'rb\') as stream:\n                    if first:\n                        start = max(0, stat.st_size - 128 * 1024)\n                        stream.seek(start)\n                        if start:\n                            stream.readline()  # discard a partial leading line\n                    else:\n                        stream.seek(state[\'offset\'])\n                    chunk = stream.read()\n                    state[\'offset\'] = stream.tell()\n                lines = (state[\'pending\'] + chunk).split(b\'\\n\')\n                state[\'pending\'] = lines.pop()\n                events = [event for line in lines if (event := metric_event(line.decode(\'utf-8\', errors=\'replace\'), state[\'stage\']))]\n                if first:\n                    # Show the latest stored values without replaying every old\n                    # epoch on resume. New appended events are emitted in full.\n                    latest = {event[0]: (i, event) for i, event in enumerate(events)}\n                    events = [event for i, event in sorted(latest.values())]\n                    if not events:\n                        print(f"[GPU {state[\'gpu\']} | {state[\'job\']}] Waiting for the first completed epoch; metrics are not available yet.", flush=True)\n                for event in events:\n                    self.emit(state, event, historical=first and state[\'has_history\'])\n                    changed = True\n                state[\'initialized\'] = True\n            summary = self.root / key / \'summary.json\'\n            if summary.is_file() and not state[\'summary_seen\']:\n                measured = json.loads(summary.read_text())\n                score = \'val_acc\' if state[\'stage\'] == \'tune\' else \'test_acc\'\n                fields = {score: measured.get(\'AA\'), \'AIA\': measured.get(\'AIA\'),\n                          \'forgetting\': measured.get(\'forgetting\'), \'gpu_hours\': measured.get(\'gpu_hours\')}\n                self.emit(state, (\'COMPLETE\', fields, json.dumps(fields)), historical=state[\'summary_history\'])\n                state[\'summary_seen\'] = True\n                changed = True\n        if changed:\n            target = self.root / \'performance_latest.json\'\n            temporary = target.with_suffix(\'.json.tmp\')\n            snapshot = {key: {k: state[k] for k in (\'gpu\', \'job\', \'stage\', \'latest\')} for key, state in self.states.items()}\n            temporary.write_text(json.dumps(snapshot, indent=2, allow_nan=False) + \'\\n\')\n            os.replace(temporary, target)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--root\', required=True)\n    parser.add_argument(\'--jobs\', required=True)\n    parser.add_argument(\'command\', nargs=argparse.REMAINDER)\n    args = parser.parse_args()\n    command = args.command[1:] if args.command[:1] == [\'--\'] else args.command\n    if not command:\n        parser.error(\'Provide the unchanged launcher command after --\')\n    follower = MetricFollower(args.root, json.loads(Path(args.jobs).read_text()))\n    print(\'Live metrics: train loss/accuracy after each epoch; evaluation accuracy/forgetting after each task. \'\n          \'Clock shows observer elapsed time. Existing log values are labeled previous log.\', flush=True)\n    stop = threading.Event()\n    def watch():\n        while not stop.is_set():\n            try:\n                follower.poll()\n            except Exception as error:\n                # The observer must never stop or alter the training process.\n                print(\'PERFORMANCE OBSERVER WARNING:\', str(error), flush=True)\n            stop.wait(2)\n    thread = threading.Thread(target=watch, daemon=True)\n    thread.start()\n    try:\n        # Keep launcher\'s stdout/heartbeat and environment/signal behavior.\n        code = subprocess.call(command)\n    finally:\n        stop.set()\n        thread.join(timeout=5)\n        try:\n            follower.poll()\n        except Exception as error:\n            print(\'PERFORMANCE OBSERVER WARNING:\', str(error), flush=True)\n    raise SystemExit(code if code >= 0 else 128 - code)\n\n\nif __name__ == \'__main__\':\n    main()\n'
PERFORMANCE_OBSERVER_SHA256 = '67b60f77281762d8f5a44444f8d3b7e2d99f1255d976e60a04bc9ea5ac0eaa07'

ENV['GL_EXECUTION_PROFILE']=str(OUTPUT/'execution_profile.json')
# Also enforce the gate here if this launch cell is run manually.
if STAGE == 'tune':
    subprocess.run([PYTHON, 'scripts/benchmark_throughput.py', '--verify-only',
                    '--data', DATA_ROOT, '--pretrained', PRETRAINED,
                    '--out', str(OUTPUT/'throughput_probe')], env=ENV, check=True)
DEADLINE = SESSION_START + 11*3600  # includes environment setup; 5-min hard grace
jobs_path = RUN_ROOT/'effective_jobs'/f'{STAGE}.json'
atomic_json(JOBS, jobs_path)
atomic_json(dict(run_tag=RUN_TAG, stage=STAGE, seed=0, source_commit=CODE_COMMIT,
                 jobs=JOBS, experiment_protocol='amp_fp16_resize256_tune10_v2', workers=2, OMP_NUM_THREADS=1, MKL_NUM_THREADS=1,
                 performance_observer_sha256=globals().get('PERFORMANCE_OBSERVER_SHA256'),
                 protocol='Full registered horizon; exit75 resumes without truncation'),
            OUTPUT/f'notebook_protocol_{STAGE}.json')
command = [PYTHON, '-m', 'glcl.kaggle', '--stage', STAGE, '--root', str(OUTPUT),
           '--data', DATA_ROOT, '--pretrained', PRETRAINED, '--deadline', str(DEADLINE), '--jobs', str(jobs_path)]
observer = [PYTHON, '-u', '-c', PERFORMANCE_OBSERVER, '--root', str(OUTPUT), '--jobs', str(jobs_path), '--', *command]
result = subprocess.run(observer, env=ENV)
if result.returncode not in (0, 75):
    raise RuntimeError('Run failed. Inspect glcl_outputs/*/*/console.log')
print('PAUSED: attach this Output and rerun same notebook' if result.returncode == 75 else 'Stage complete')

In [ ]:
for path in (OUTPUT/'selection.json', OUTPUT/'session.json', OUTPUT/'comparison.csv', OUTPUT/'comparison.md'):
    if path.exists():
        print(path.name, path.read_text())
print('Save this notebook Output. Attach its glcl_outputs tree to resume; no tar backup is required.')
print('If resume_dependencies.json names earlier Output shards, keep those versions attached too.')